# 09 · Gaussian backend benchmarks

Four classic results of Gaussian quantum optics, each computed three ways where possible:
**closed form**, the **dense Fock oracle** (`qttlind.reference.cv_dense`: gates as `expm`, truncated
Fock space) and **the Gaussian backend** (`qttlind.gaussian`).

1. Photon statistics of squeezed light (only even photon numbers; closed form).
2. Squeezing degraded by loss: $V_x=\eta e^{-2r}/2+(1-\eta)/2$ — the dB budget of a real source.
3. Entanglement of a two-mode squeezed vacuum under loss: it survives any finite loss
   (logarithmic negativity, checked against the dense partial transpose).
4. Gaussian boson sampling: pattern probabilities are loop hafnians (Hamilton *et al.*,
   PRL 119, 170501, 2017), and click probabilities Torontonians (Quesada *et al.*, PRA 98, 062322, 2018).

The last cell is an **acceptance table**: every deviation must be below its tolerance.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tutorial_utils as tu
from tutorial_utils import cd, dr

print(tu.cv_banner())
if tu.HAVE_GAUSSIAN:
    from qttlind.gaussian import GaussianState, ops, measure as gm, fock as gf, hafnian as hf
checks = {}   # name -> (deviation, tolerance)

## 1. Squeezed vacuum: only pairs of photons

$\langle 2m|S(r)|0\rangle = (-\tanh r)^m\,\frac{\sqrt{(2m)!}}{2^m m!}\,\frac{1}{\sqrt{\cosh r}}$, odd numbers vanish:
squeezing creates photons in pairs.

In [ ]:
r, nmax = 0.8, 16
closed = np.abs(cd.squeezed_vacuum_amplitudes(nmax, r)) ** 2
dense = np.abs(tu.DenseModes([80]).gate("S", (0,), r).psi[:nmax]) ** 2
checks["squeezed vacuum: oracle vs closed form"] = (np.abs(dense - closed).max(), 1e-10)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(np.arange(nmax) - 0.2, closed, 0.4, label="closed form")
ax.bar(np.arange(nmax) + 0.2, dense, 0.4, label="dense oracle")
if tu.HAVE_GAUSSIAN:
    s = GaussianState.squeezed_vacuum(r)
    mine = np.array([hf.pattern_probability(s, [n]) for n in range(nmax)])
    rho = gf.fock_density_matrix(s, nmax)
    ax.plot(np.arange(nmax), mine, "k.", label="qttlind (hafnian)")
    checks["squeezed vacuum: hafnian vs closed form"] = (np.abs(mine - closed).max(), 1e-10)
    checks["squeezed vacuum: Bargmann vs closed form"] = (np.abs(np.diag(rho).real - closed).max(), 1e-12)
ax.set_xlabel("n"); ax.set_ylabel("P(n)"); ax.set_yscale("log"); ax.set_ylim(1e-6, 1); ax.legend()
ax.set_title(f"squeezed vacuum, r = {r} ({20 * r * np.log10(np.e):.1f} dB)")
tu.savefig(fig, "09_squeezed_statistics.png");

## 2. Squeezing under loss

A squeezed state that loses a fraction $1-\eta$ of its light mixes with vacuum:
$V_x = \eta\,e^{-2r}/2 + (1-\eta)/2$. In dB below shot noise, $-10\log_{10}(2V_x)$: even perfect
squeezing cannot beat $-10\log_{10}(1-\eta)$ — the reason why optical losses dominate the error
budget of CV hardware.

In [ ]:
etas = np.linspace(0.3, 1.0, 15)
fig, ax = plt.subplots(figsize=(6, 3.5))
for r in (0.5, 1.0, 1.5, 3.0):
    Vx = etas * np.exp(-2 * r) / 2 + (1 - etas) / 2
    ax.plot(etas, -10 * np.log10(2 * Vx), label=f"r = {r}")
    if tu.HAVE_GAUSSIAN:
        mine = [ops.loss(GaussianState.squeezed_vacuum(r), e).cov[0, 0] for e in etas]
        ax.plot(etas, -10 * np.log10(2 * np.array(mine)), "k.", ms=3)
ax.plot(etas, -10 * np.log10(1 - etas + 1e-12), "k--", label="limit r -> inf")
ax.set_xlabel("transmissivity eta"); ax.set_ylabel("squeezing (dB)"); ax.set_ylim(0, 16); ax.legend()
tu.savefig(fig, "09_squeezing_loss.png")

# oracle check at one point: Kraus loss channel on the dense state
N, r, eta = 60, 0.5, 0.7
k = tu.DenseModes([N]).gate("S", (0,), r).psi
_, V = cd.moments(cd.loss_channel(np.outer(k, k.conj()), eta, 0, [N]), [N])
checks["lossy squeezing: oracle vs closed form"] = (abs(V[0, 0] - (eta * np.exp(-2 * r) / 2 + (1 - eta) / 2)), 1e-9)

## 3. Two-mode squeezing: entanglement survives any finite loss

$E_N = \sum_k \max(0, -\log_2 2\tilde\nu_k)$ from the partially transposed covariance matrix. For the
two-mode squeezed vacuum $E_N = 2r/\ln 2$; with loss $\eta$ on one arm it decreases but stays
positive for every $\eta > 0$. The oracle computes $\log_2\|\rho^{T_B}\|_1$ from the dense state.

In [ ]:
r = 0.5
etas = np.linspace(0.02, 1.0, 25)
N = 22
k = tu.DenseModes([N, N]).gate("S2", (0, 1), r).psi
rho0 = np.outer(k, k.conj())

def dense_log_neg(rho):
    rt = rho.reshape(N, N, N, N).transpose(0, 3, 2, 1).reshape(N * N, N * N)
    return np.log2(np.abs(np.linalg.eigvalsh((rt + rt.conj().T) / 2)).sum())

dense_pts = [(e, dense_log_neg(cd.loss_channel(rho0, e, 1, [N, N]))) for e in (0.1, 0.5, 1.0)]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(*zip(*dense_pts), "o", label="dense oracle")
if tu.HAVE_GAUSSIAN:
    s0 = ops.two_mode_squeeze(GaussianState.vacuum(2), r)
    mine = [gm.log_negativity(ops.loss(s0, e, mode=1), [0]) for e in etas]
    ax.plot(etas, mine, label="qttlind")
    dev = max(abs(gm.log_negativity(ops.loss(s0, e, mode=1), [0]) - v) for e, v in dense_pts)
    checks["log-negativity under loss: backend vs oracle"] = (dev, 1e-6)
    checks["TMSV log-negativity = 2r/ln2"] = (abs(gm.log_negativity(s0, [0]) - 2 * r / np.log(2)), 1e-12)
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("transmissivity of arm B"); ax.set_ylabel("E_N (ebits)")
ax.legend(); tu.savefig(fig, "09_tmsv_loss.png");

## 4. Gaussian boson sampling

Four single-mode squeezed vacua ($r = 0.6$) enter a Haar-random $4\times4$ interferometer. The
probability of a photon pattern $n$ is $p(n) = e^{c}\,\mathrm{lhaf}(\tilde A_n)/\prod_k n_k!$. The oracle
starts from the *exact* (truncated) squeezed amplitudes, applies the interferometer by `expm` in a
$7^4$-dimensional Fock space and reads $|\langle n|\psi\rangle|^2$. The interferometer conserves the
photon number, so every pattern with at most 6 photons is exact in the oracle; probabilities of
**click** patterns sum over all photon numbers and are only correct up to the oracle's missing
tail, which sets their tolerance.

(Squeezing the vacuum by `expm` in 7 levels would be badly truncated at $r=0.6$ — try it.)

In [ ]:
rng = np.random.default_rng(7)
Z = rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4))
U, _ = np.linalg.qr(Z)
r, cut = 0.6, 7
t0 = time.perf_counter()
sq = cd.squeezed_vacuum_amplitudes(cut, r)  # exact amplitudes, truncated (not renormalised)
dm4 = tu.DenseModes([cut] * 4).product_state(sq, sq, sq, sq).interferometer(U)
P = dm4.probabilities()
tail = 1 - P.sum()
print(f"dense oracle: {time.perf_counter() - t0:.1f} s for a {cut**4}-dimensional space; missing tail {tail:.1e}")

patterns = [n for n in np.ndindex(*([4] * 4)) if sum(n) <= 4]
patterns.sort(key=lambda n: -P[n])
top = patterns[:16]
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(top)), [P[n] for n in top], label="dense oracle")
if tu.HAVE_GAUSSIAN:
    s = ops.interferometer(GaussianState.product(*[GaussianState.squeezed_vacuum(r)] * 4), U)
    mine = [hf.pattern_probability(s, list(n)) for n in top]
    ax.plot(range(len(top)), mine, "k.", label="qttlind (loop hafnian)")
    checks["GBS pattern probabilities"] = (max(abs(m - P[n]) for m, n in zip(mine, top)), 1e-10)
    clicks = [(0, 1, 0, 1), (1, 1, 0, 0), (1, 1, 1, 1)]
    def dense_click(c):
        mask = [(np.arange(cut) > 0) == bool(x) for x in c]
        return P[np.ix_(*mask)].sum()
    checks["GBS click probabilities"] = (max(abs(hf.click_probability(s, list(c)) - dense_click(c)) for c in clicks), tail + 1e-12)
ax.set_xticks(range(len(top)), ["".join(map(str, n)) for n in top], rotation=60)
ax.set_ylabel("probability"); ax.legend(); tu.savefig(fig, "09_gbs.png");

### Hafnian cost

Power traces cost $O(n^3 2^n)$ for a $2n\times 2n$ matrix, the definition $(2n-1)!!$ terms.

In [ ]:
if tu.HAVE_GAUSSIAN:
    sizes, fast, brute = [4, 6, 8, 10, 12], [], []
    for m in sizes:
        A = rng.normal(size=(m, m)); A = A + A.T
        t0 = time.perf_counter(); h1 = hf.hafnian(A); fast.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); h2 = cd.hafnian_recursive(A - np.diag(np.diag(A))); brute.append(time.perf_counter() - t0)
        checks[f"hafnian {m}x{m} vs brute force"] = (abs(h1 - h2) / max(1, abs(h2)), 1e-9)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.semilogy(sizes, fast, "o-", label="power traces"); ax.semilogy(sizes, brute, "s-", label="definition")
    ax.set_xlabel("matrix size"); ax.set_ylabel("seconds"); ax.legend(); tu.savefig(fig, "09_hafnian_cost.png")
else:
    print("qttlind.gaussian is not available: hafnian timing skipped.")

## Acceptance table

In [ ]:
for name, (dev, tol) in checks.items():
    print(f"{'ok ' if dev <= tol else 'FAIL'} {name:<48s} deviation {dev:9.2e}  (tolerance {tol:.0e})")
assert all(dev <= tol for dev, tol in checks.values()), "a benchmark is outside its tolerance"